# Ad Source Contribution Analysis

Measures how much each advertising channel contributes to signed contracts: leads are cleaned,
merged with contracts on email, and monthly contribution rates are computed per site over a
rolling three-month window. Overall conversion rates are reported by channel.

Stack: pandas, PySpark, MLflow experiment tracking, Delta Live Tables.

Note: company names, site IDs, and internal paths are anonymized, and outputs are cleared
because the originals ran on internal data.


In [ ]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
import mlflow

In [ ]:
# Set an experiment name
experiment_name = f"/Repos/analyst/Ad Contribution Rate/DataScience/ad_source_contribution_rate_01"

# Check if the experiment already exists
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(experiment_name)

# Set the experiment to use
mlflow.set_experiment(experiment_name)

# Start an MLflow run and get the run ID
run = mlflow.start_run()
run_id = run.info.run_id

# Set the run name
run_name = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
mlflow.set_tag("mlflow.runName", run_name)


# Data Loading

In [ ]:
# Reading the leads data from cloud

# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()

# Run the SQL Query
query = """
SELECT
 Site,
 Inquiry_ID,
 Rental_Office_Email,
 Created_Date,
 Suite_Type,
 Phone_Number,
 Ad_Source,
 Date_Loaded,
 Email,
 Valid
FROM hive_metastore.leads.email_inquiries 
"""

# execute the query and load data into a PySpark DataFrame
raw_data_leads_spark = spark.sql(query)

# convert the Spark DataFrame to a pandas DataFrame
leads = raw_data_leads_spark.toPandas()

# Load Hierarchy map data
hierarchy_map = pd.read_csv("/dbfs/FileStore/ad_source/Hierarchy_Sep11_2024.csv") 

In [ ]:
# Load contracts data
df0 = pd.read_csv("/dbfs/mnt/adls-bidaadlsprd1/contracts/BW0BPARTNER$P.txt", delimiter='|')
df1 = pd.read_csv("/dbfs/mnt/adls-bidaadlsprd1/contracts/BW0RECONTRACT$P.txt", delimiter='|', low_memory=False) 
df2 = pd.read_csv("/dbfs/mnt/adls-bidaadlsprd1/contracts/BW0REFXDS02$F.txt", delimiter='|', low_memory=False)
df3 = pd.read_csv("/dbfs/mnt/adls-bidaadlsprd1/contracts/BWZRENTCON$Q.txt", delimiter='|', low_memory=False)
df4 = pd.read_csv("/dbfs/mnt/adls-bidaadlsprd1/contracts/BWZRECNTRT$F.txt", delimiter='|', low_memory=False)


In [ ]:
df0.sample(3)

In [ ]:
# BW0RECONTRACT$P
df1.sample(3)

In [ ]:
# BW0REFXDS02$F
df2.sample(3)

In [ ]:
# BW0RENTOBJECT$P
df3.sample(3)

In [ ]:
# keeping the useful features
df1_contracts = df1[['COMP_CODE', 'RECONTRACT', '/BIC/ZDERF', 'CONTRSTART', 'CONTRTYPE']].dropna(subset=['RECONTRACT'])
df1_contracts = df1_contracts.query('CONTRTYPE == "ZO01" or CONTRTYPE == "ZLRO"').copy() # Filter out the parking lease
df2_bpartner = df2[['RECONTRACT', 'BPARTNER']].dropna(subset=['RECONTRACT'])
df3_site = df3[['COMP_CODE', 'RECONTRACT', 'BUSENTITY']].dropna(subset=['RECONTRACT']).drop_duplicates(keep='last')
df4_initial = df4[['RECONTRACT', 'COMP_CODE','/BIC/ZCNUSRSTA']].copy()

# Merging the datasets to include the contracts with no email
emails = df0.merge(df2_bpartner, on='BPARTNER', how='right')
contracts = df1_contracts.merge(df3_site, on=['RECONTRACT', 'COMP_CODE'], how='inner')
contracts = contracts.merge(df4_initial, on=['RECONTRACT', 'COMP_CODE'], how='inner')
final_contracts = contracts.merge(emails, on=['RECONTRACT'], how='left')

# Data Cleaning

### Contract

In [ ]:
# Rename some columns
df_contracts = (final_contracts.rename(columns={'RECONTRACT': 'Lease',
                                       '/BIC/ZDERF':'contract_creation_date',
                                       'CONTRSTART' : 'contract_start_date',
                                       'EMAIL_ADDR': 'contract_email',
                                       'BUSENTITY':'contract_site',
                                       'BPARTNER':'business_partner',
                                       '/BIC/ZCNUSRSTA':'contract_status'})).copy()

# converting the date columns to datetime format
df_contracts['contract_creation_date'] = pd.to_datetime(df_contracts['contract_creation_date'])
df_contracts['contract_start_date'] = pd.to_datetime(df_contracts['contract_start_date'])

# lower case emails
df_contracts['contract_email']=df_contracts.contract_email.str.lower()

In [ ]:
# Drop na Business Partner values
df_contracts.dropna(subset = 'business_partner', inplace=True)
df_contracts.sort_values(by='contract_creation_date', ascending=True,inplace=True)

In [ ]:
# Fill the auto filled emails with nan values

# Patterns to look for in the 'contract_email' field
patterns = ['noemail', 'no@email.com', 'no@gmail.com']

# Use .loc[] to locate rows matching the patterns and assign NaN values
for pattern in patterns:
    df_contracts.loc[df_contracts['contract_email'].str.contains(pattern, na=False), 'contract_email'] = np.nan


## Remove the auto filled emails
# df_contracts = df_contracts.apply(lambda row: row[~(df_contracts['contract_email'].str.contains('noemail') 
#                                               | df_contracts['contract_email'].str.contains('no@email.com') 
#                                               | df_contracts['contract_email'].str.contains('no@gmail.com'))])


In [ ]:
# Remove duplicate contract records
df_contracts = df_contracts.drop_duplicates()

In [ ]:
# Filter out the contracts with initial status
df_contracts = df_contracts[df_contracts.contract_status != 'Initial']

In [ ]:
# Hierarchy Map Data
cols = ['Acquisition Date', 'BPC_CC', 'Country', 'City', 'Building Type', 'Building Class', 'Region', 'Site', 'Director','Operations Manager']

hierarchy_map = hierarchy_map[cols].drop_duplicates().copy()
hierarchy_map.dropna(subset=['Site'], inplace=True)
hierarchy_map['Site'] = hierarchy_map['Site'].str.split('/').str[1].astype(int)
hierarchy_map['Acquisition Date'] = pd.to_datetime(hierarchy_map['Acquisition Date'])

# Merging the contract_creation_dates and hierarchy map to get the aquizition feature
df_contracts = df_contracts.merge (
    right =hierarchy_map,
    how='left',
    left_on= 'contract_site',
    right_on ="Site"
)

# keep only Canada contracts
df_contracts = df_contracts[df_contracts.Country=='CA']

In [ ]:
# Only consider the contracts that has creation date after the aquistion date

# Number of contracts that are excluded by applying this rule
# Track the number of contracts excluded
num_excluded = df_contracts[df_contracts['Acquisition Date'] >= df_contracts['contract_start_date']]['Lease'].nunique()
print('Number of contracts transfered by aqquizition: ',num_excluded)

# Excluding the contracts that came with aquizition
df_contracts = df_contracts[df_contracts['Acquisition Date'] < df_contracts['contract_start_date']]

# logging metrics
mlflow.log_metric("num_excluded_contracts", num_excluded)   # Track metrics
mlflow.log_metric('df_contracts_length', df_contracts.shape[0])  # Track metrics

### Leads

In [ ]:
# Drop unnecessary columns
df_leads = leads.drop(["Phone_Number", "Rental_Office_Email"], axis = 1)

# Change the date columns to datetime format
df_leads['Date_Loaded'] = pd.to_datetime(df_leads['Date_Loaded'])
df_leads['Created_Date'] = pd.to_datetime(df_leads['Created_Date'])

# rename some columns to more appropriate name
df_leads.rename(columns={'Created_Date':'leads_creation_date', 'Site':'lead_site'},inplace=True)

In [ ]:
# checking for duplicates (keep last) based on inquiry ID
df_leads.sort_values(by='Date_Loaded',inplace=True)
df_leads.drop_duplicates(subset=['Inquiry_ID'],keep='last',inplace=True)

In [ ]:
## drop rows where Ad_Source or Email is NA
df_leads.dropna(subset=['Ad_Source'],inplace=True)
df_leads.dropna(subset=['Email'],inplace=True)

In [ ]:
# Dealing with spelling errors in ad sources
df_leads["Ad_Source"] = df_leads["Ad_Source"].str.lower() # lower cas the ad source names

df_leads['Ad_Source'] = df_leads['Ad_Source'].replace({'facebok':'facebook',
                                                       'companywebst.com':'company-website.ca',
                                                       'rental-site.com':'rental-site.ca',
                                                       'logisqu�bec':'LogisQuebec',
                                                       'logisquébec': 'LogisQuebec'
                                                       }) # replace the typos with the correct names
print(len(df_leads.loc[df_leads["Ad_Source"]=='facebok']))
print(len(df_leads.loc[df_leads["Ad_Source"]=='companywebst.com']))
print(len(df_leads.loc[df_leads["Ad_Source"]=='rental-site.com']))

In [ ]:
# Cleaning Autofilled Null Email Address
df_leads["Email"] = df_leads["Email"].str.lower()
df_leads = df_leads.rename(columns={"Email": "lead_email"})
# looking at autofilled emails in leads
na_email = df_leads.lead_email.str.match('^(noemail)')
df_leads = df_leads[~na_email]

In [ ]:
# # remove 3rd party sources
# company_ad_sources = ["company-website.ca", "partner-site.ca"]
# df_leads = df_leads[~df_leads['Ad_Source'].isin(company_ad_sources)]

In [ ]:
# Filter to only leads after 2021-01-01 and valid
filter = (df_leads['leads_creation_date']>='2021-01-01')
df_leads = df_leads.loc[filter]
df_leads = df_leads[df_leads.Valid]

# Continue logging to the same run
mlflow.log_metric('df_leads_length', df_leads.shape[0]) # Track metrics

### Merging the Contract emails with Leads and Cleaning

In [ ]:
# Merging leads and contracts dataframes on emails
df_final = pd.merge(
    left = df_leads,
    right = df_contracts, 
    how = "left",
    left_on='lead_email',
    right_on='contract_email'
)

In [ ]:
# Filter out lead created date is before contract creation date and keep null creation date in contracts (This is not converted people)
df_final = df_final[((df_final['leads_creation_date'])<=df_final["contract_creation_date"]) | (df_final["contract_creation_date"].isnull())]

In [ ]:
# Remove the leads for which the time interval to contract creation is more than 90 days 

# set interval
interval = 180

# Create new column that has difference in days: formula -> (Contract Creation Date - Lead Creation Date)
df_final['difference_days'] = (df_final['contract_creation_date'] - df_final['leads_creation_date']).dt.days

# update to show only rows within the 90 interval or null
df_final=df_final[df_final['difference_days'].isna() | (df_final['difference_days'] <= interval)]

In [ ]:
# Remove duplicates for same email and same ad source
df_final.sort_values(by=['lead_email','Ad_Source', 'leads_creation_date', 'Lease'], inplace=True)
df_final.drop_duplicates(subset=['lead_email','Ad_Source', 'Lease'], keep='last',inplace=True)

# Overall Conversion Rate

In [ ]:
df_conversion = df_final.groupby(by=["Ad_Source"], as_index=False).count().sort_index()

# Renaming col to appropriate names
df_conversion.rename(columns={"contract_email": "Conversions", "lead_email": "Leads"}, inplace=True)

# Calculating conversion rate for each ad source
df_conversion["Conversion Rate"] = df_conversion["Conversions"] / df_conversion["Leads"]


# Calculating conversion rate for each ad source
df_conversion["Conversion Rate"] = (df_conversion["Conversions"] / df_conversion["Leads"] * 100).round(2)

# take only useful columns
df_conversion = df_conversion[["Ad_Source","Conversions", "Leads", "Conversion Rate"]].sort_index()
df_conversion.sort_values(by='Conversions', ascending=False, inplace=True)
df_conversion.reset_index(drop=True, inplace=True)
df_conversion

# Overall Ad Source Contribution Rate

In [ ]:
# only keep converged leads
df_converged = df_final[df_final['contract_email'].notna()].copy()

# Calculating the contrbution rate for each ad source
df_contribution = df_converged.Ad_Source.value_counts().to_frame()
df_contribution.rename(columns={'Ad_Source':'conversion'}, inplace=True)
df_contribution.index.name = 'Ad_source'
df_contribution['contribution_rate'] = (df_converged.Ad_Source.value_counts()/df_converged['Lease'].nunique()).round(3)
df_contribution['total_contracts'] = df_converged['Lease'].nunique()
df_contribution = df_contribution.reset_index()
df_contribution.rename(columns={'count':'conversion'}, inplace=True)
df_contribution

# Monthly Contribution Rate
For each site, calculate the contribution rate of each ad source in each month considering leads generated in the last three months

In [ ]:
import pandas as pd
from collections import defaultdict

# Output table dictionaries
output_table_total_contracts = defaultdict(dict)

# Output list for long format
output_total_contracts_list = []
output_total_online_contracts_list = []
output_conversions_list = []
output_contribution_rate_list = []

# Iterate over each unique month-year in contracts dataframe
for month_year in df_contracts['contract_creation_date'].dt.to_period('M').unique():
    
    # Filter contracts for the current month_year
    contracts_current_month = df_contracts[df_contracts['contract_creation_date'].dt.to_period('M') == month_year]
    
    
    # Filter leads for the leads up to the end of current month
    end_date = pd.to_datetime(str(month_year)+ "-01") + pd.DateOffset(months=1) - pd.DateOffset(days=1)
    leads_filtered = df_leads[(df_leads['leads_creation_date'] <= end_date)]
    
    # Merge on email to get the conversions
    merged_df = pd.merge(contracts_current_month, leads_filtered, left_on='contract_email', right_on='lead_email')
        
    # Filter out lead created date is before contract creation date and keep null creation date in contracts (these are not converted leads)
    merged_df = merged_df[((merged_df['leads_creation_date'])<=merged_df["contract_creation_date"]) | (merged_df["contract_creation_date"].isnull())].copy()

    # Remove the leads for which the time interval to contract creation is more than 180 days 
    # Create new column that has difference in days: formula -> (Contract Creation Date - Lead Creation Date)
    merged_df['difference_days'] = (merged_df['contract_creation_date'] - merged_df['leads_creation_date']).dt.days
    # set interval
    interval = 180
    # update to show only rows within the 180 interval or null
    merged_df=merged_df[merged_df['difference_days'].isna() | (merged_df['difference_days'] <= interval)]
    
    # Remove duplicates for the same ad source under the same lease
    # (Count one lead per ad source per lease)
    merged_df.drop_duplicates(subset=['lead_email','Ad_Source', 'Lease'],keep='last',inplace=True)

    # Calculate the total contracts and the contracts without email for the current month for each site
    for site in df_contracts['contract_site'].unique():
        site_contracts_current_month = contracts_current_month[contracts_current_month.contract_site == site]
        site_total_contracts = site_contracts_current_month['Lease'].nunique()
        site_contracts_current_month = site_contracts_current_month.sort_values(by=['Lease', 'contract_email'])
        site_contracts_current_month = site_contracts_current_month.groupby('Lease').agg({'contract_site':'first',
                                                            'contract_email':'first'}).reset_index()
        site_non_email_contracts = site_contracts_current_month[(~site_contracts_current_month.Lease.isin(merged_df[merged_df['contract_site'] == site]['Lease'].unique()))&
                                                                (site_contracts_current_month.contract_email.isna())]['Lease'].nunique()
        
        output_total_contracts_list.append({'site':site,'date':str(month_year), 
                                            'non_email_contracts': site_non_email_contracts,
                                            'total_contracts':site_total_contracts})

    # Calculate the total online contracts for the current month for each site
    for site in merged_df['contract_site'].unique():
        site_merged_df = merged_df[merged_df['contract_site'] == site]
        site_total_online_contracts = site_merged_df['Lease'].nunique()
        output_total_online_contracts_list.append({'site':site,'date':str(month_year), 'total_online_contracts':site_total_online_contracts})

        # For each site calculate the conversion and contribution rate for each ad_source
        for ad_source in df_leads['Ad_Source'].unique():
            conversions = len(site_merged_df[site_merged_df['Ad_Source'] == ad_source])
            contribution_rate = (conversions/site_total_online_contracts)
            output_conversions_list.append({'site':site,'date':str(month_year), 'ad_source':ad_source, 'conversions':conversions})
            output_contribution_rate_list.append({'site':site,'date':str(month_year), 'ad_source':ad_source, 'contribution_rate':contribution_rate})

# Convert the output dictionaries to DataFrames
output_df_conversions = pd.DataFrame(output_conversions_list)
output_df_rate = pd.DataFrame(output_contribution_rate_list)
output_df_total_online_contracts = pd.DataFrame(output_total_online_contracts_list)
output_df_total_contracts = pd.DataFrame(output_total_contracts_list)
output_df_contracts = output_df_total_contracts.merge(output_df_total_online_contracts, on=['site', 'date'], how='left').fillna(0)

### Create Delta Live Tables 

In [ ]:
# Convert the Pandas DataFrame to a PySpark DataFrame
output_df_conversions_spark = spark.createDataFrame(output_df_conversions)
output_df_rate_spark = spark.createDataFrame(output_df_rate)
output_df_total_contracts_spark = spark.createDataFrame(output_df_contracts)

# Write the PySpark DataFrame as a Delta table
table_name = "leads.online_leads_conversions"
output_df_conversions_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

table_name = "leads.online_leads_contribution_rate"
output_df_rate_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

table_name = "leads.total_contracts"
output_df_total_contracts_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

In [ ]:
mlflow.end_run()

# Validation

In [ ]:
s = 11001
y = 2023
m = '10'

display((df_final[(df_final.contract_site==s)&
         (df_final.contract_creation_date.dt.year==y)&
         (df_final.contract_creation_date.dt.month==int(m))
         ][['Ad_Source', 'lead_site', 'lead_email', 'leads_creation_date', 'contract_email','contract_creation_date', 'contract_site', 'business_partner','Lease']]))

print('-----------------------------------------------')
print('contribution of each ad source:')
display((df_final[(df_final.contract_site==s)&
         (df_final.contract_creation_date.dt.year==y)&
         (df_final.contract_creation_date.dt.month==int(m))
         ][['Ad_Source', 'lead_site', 'lead_email', 'leads_creation_date', 'contract_email','contract_creation_date', 'contract_site', 'business_partner','Lease']].
 groupby('Ad_Source')['Lease'].nunique()))

print('-----------------------------------------------')
t = df_final[(df_final.contract_site==s)&
         (df_final.contract_creation_date.dt.year==y)&
         (df_final.contract_creation_date.dt.month==int(m))
         ]['Lease'].nunique()
print('total_online_contracts= ', t)
print('-----------------------------------------------')
print('contribution rate of each ad source:')
d = str(y)+'-'+m
display(output_df_rate[(output_df_rate.site==s)&
               (output_df_rate.date==d)].query('contribution_rate>0'))        

print('-----------------------------------------------')
print('total contracts:')
output_df_contracts[(output_df_contracts.site==s)&
                    (output_df_contracts.date==d)]

In [ ]:
output_df_contracts

In [ ]:
s = 11002
y = 2022
output_df_contracts[(output_df_contracts.site==s)&
                    (pd.to_datetime(output_df_contracts.date).dt.year==y)][['total_contracts','non_email_contracts', 'total_online_contracts']].sum()

In [ ]:
output_df_contracts[(output_df_contracts.site==s)][['total_contracts','non_email_contracts', 'total_online_contracts']].sum()

In [ ]:
output_df_contracts[pd.to_datetime(output_df_total_contracts.date).dt.year==2023][['total_contracts','total_online_contracts','non_email_contracts']].sum()

In [ ]:
(output_df_conversions[(output_df_conversions.site==s)&
                      (output_df_conversions.date==str(y)+'-'+m)]).sort_values('conversions', ascending=False)